# TabPFN vs. gradient boosting, with and without the news-opposition score

Runs in **Google Colab on a GPU** (Runtime → Change runtime type → T4 GPU). TabPFN on ~4,000 training rows is too slow on a laptop CPU.

The question, in course terms:
- **Lecture 8** found that neural nets rarely beat tuned tree ensembles on tabular data.
- **Lecture 12** showed that pre-trained transformers can win where data is scarce.

TabPFN is a transformer pre-trained on millions of synthetic tables, which makes it a direct test of both claims. The `--ablation` flag runs each model with and without the text features, which measures what the news model adds.

**One-time setup, in the Colab sidebar 🔑 (Secrets):**
- `GITHUB_TOKEN`: a GitHub personal access token with read access to the private repo.
- `TABPFN_TOKEN`: from https://ux.priorlabs.ai, after accepting the model license (non-commercial use only).

**Validation only.** The test cohort is scored only with `--final`, once, when the team agrees.

In [ ]:
from google.colab import userdata
import os, subprocess
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("queue-risk-ai"):
    subprocess.run(["git", "clone", f"https://{token}@github.com/devault-settings/queue-risk-ai.git"], check=True)
%cd queue-risk-ai
!git pull -q

In [ ]:
!pip install -q -e . tabpfn
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")

Optional: the opposition score table is gitignored. If the NLP workstream has produced it, upload `opposition_scores.csv` here. Otherwise the ablation reports that there are no text features yet.

In [ ]:
from google.colab import files
os.makedirs("data/processed", exist_ok=True)
try:
    up = files.upload()            # choose opposition_scores.csv, or cancel
    for name, content in up.items():
        open(f"data/processed/{name}", "wb").write(content)
except Exception as e:
    print("skipped:", e)

In [ ]:
!python tests/selftest.py

In [ ]:
!python -m queue_risk.pipeline --models "Historical,Gradient,TabPFN" --ablation

In [ ]:
import pandas as pd
pd.read_csv("reports/results_within_validation.csv")